In [ ]:
# Lab type: prompt
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Auditing AI-Generated Forecasts — Capstone
# Task: Direct an AI assistant to build a forecasting evaluation, paste its output
#       below unmodified, then audit it against the five-check protocol. The audit
#       cell is where the learning happens — wrong AI output is expected and useful.

# Capstone Lab: The Direction–Audit Loop

You will run the full professional loop from Lesson 8:

1. **Direct** — write a prompt for an AI assistant (ChatGPT, Claude, Copilot, …).
2. **Generate** — paste the assistant's code into this notebook, unmodified.
3. **Audit** — score it against the five-check protocol, with evidence.
4. **(Optional) iterate** — tighten the prompt and compare a second attempt.

**Do the task twice**: once with the deliberately *vague* prompt, once with your own
*directive* prompt. The difference between the two audits is the lesson.

## Setup: the dataset your prompt should describe

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")

orders.to_csv("daily_orders.csv")
print("Saved daily_orders.csv — attach or describe this file to your AI assistant.")

## The task (give this scenario to the AI)

> The store needs a forecast of **daily orders, 7 days ahead**, refreshed daily, to
> plan warehouse staffing. History: `daily_orders.csv` (3 years of daily order counts,
> strong weekly seasonality, upward trend, a holiday-season wave). Deliverable: Python
> code that builds a model **and reports how accurate we should expect it to be in
> production**.

**Round 1 — vague prompt.** Give the assistant only something like: *"Build a model to
forecast daily orders from this CSV and report its accuracy."* (This is what most
people type. The audit will show what it buys.)

**Round 2 — directive prompt.** Write your own prompt that constrains: the 7-day
horizon, the evaluation design, the preprocessing boundary, the baseline requirement,
and the feature-shift rule. Lesson 8's example prompt is the pattern — write yours
before peeking.

## Your prompts

**Round 1 prompt (vague, verbatim):**

*(paste here)*

**Round 2 prompt (directive, verbatim):**

*(paste here)*

## Round 1 output — paste the AI's code below, unmodified, and run it

In [ ]:
# PASTE the AI assistant's Round 1 code here, unmodified.


## The audit

Score **each round** against the five checks. For every FAIL, cite the exact line(s)
of pasted code as evidence. A check with no evidence either way is a FAIL — "the code
doesn't show me" is an audit result.

| # | Check | What to inspect |
|---|-------|-----------------|
| 1 | **Split integrity** | Any `shuffle` (incl. `train_test_split` defaults), `KFold`, `cv=<int>`, `.sample()`? Chronological block or `TimeSeriesSplit`? |
| 2 | **Feature availability** | Every lag/rolling shifted ≥ 7 (the horizon)? Any `rolling()` without `shift()`, `center=True`, `bfill()`/`interpolate()` feeding features? |
| 3 | **Preprocessing boundaries** | Scaler/imputer/encoder fit only on training data (inside each fold)? Or `fit_transform` on the full series? |
| 4 | **Baseline honesty** | Seasonal-naive (and ideally Holt-Winters) computed on the same split at the same 7-day horizon? |
| 5 | **Robustness** | Per-fold backtest table with `gap=7`? Worst fold reported? Or a single number? |

In [ ]:
# Round 1 audit — fill in each verdict with evidence (line references from the paste)
audit_round_1 = {
    "1 split integrity":        "PASS/FAIL — evidence: ...",
    "2 feature availability":   "PASS/FAIL — evidence: ...",
    "3 preprocessing boundary": "PASS/FAIL — evidence: ...",
    "4 baseline honesty":       "PASS/FAIL — evidence: ...",
    "5 robustness":             "PASS/FAIL — evidence: ...",
}
for check, verdict in audit_round_1.items():
    print(f"{check:26s} {verdict}")

## Round 2 output — paste, run, audit

In [ ]:
# PASTE the AI assistant's Round 2 code here, unmodified.


In [ ]:
# Round 2 audit
audit_round_2 = {
    "1 split integrity":        "PASS/FAIL — evidence: ...",
    "2 feature availability":   "PASS/FAIL — evidence: ...",
    "3 preprocessing boundary": "PASS/FAIL — evidence: ...",
    "4 baseline honesty":       "PASS/FAIL — evidence: ...",
    "5 robustness":             "PASS/FAIL — evidence: ...",
}
for check, verdict in audit_round_2.items():
    print(f"{check:26s} {verdict}")

## Reflection

1. Which checks did the vague prompt fail that the directive prompt passed? Which
   failures survived even the directive prompt?
2. Compare the two reported accuracies. If Round 1 reported a *better* number than
   Round 2, explain why that is expected — and which number you would give the
   warehouse-staffing stakeholder.
3. Write the one sentence you'd add to your directive prompt next time.

*(Write your answers here.)*

**Instructor note — expected failure modes for the vague prompt (Round 1):**

- `train_test_split(X, y, test_size=0.2)` with default shuffle — check 1 FAIL. Seen in
  the large majority of vague-prompt outputs.
- Lag-1/rolling features with no shift, or `rolling(...).mean()` directly — check 2
  FAIL; the 7-day horizon is almost never respected without being stated (lag_1
  appears even when the assistant is told "7 days ahead" only in prose).
- `MinMaxScaler`/`StandardScaler().fit_transform` on the full frame before splitting —
  check 3 FAIL, very common in generated sklearn boilerplate.
- No baseline of any kind — check 4 FAIL in nearly all outputs; occasionally a mean
  baseline appears, almost never a seasonal naive.
- Single split, single metric (often R², sometimes RMSE with no scale context) —
  check 5 FAIL.
- Occasional bonus failures: `fillna(method="bfill")` on features; evaluating a lag-1
  model but describing it as a 7-day forecast; an LSTM for 1,096 rows.

Round 2 with a directive prompt typically passes checks 1, 3, 4 and often 2; the most
common surviving failure is a missing `gap=7` (check 5) and lag shifts < horizon when
the assistant adds "extra" features beyond the specified ones — good discussion
material for why the audit remains necessary even with good direction.